In [1]:
from mdcrow import MDCrow
from langchain.callbacks import get_openai_callback

In [2]:
def calculate_llm_cost(input_tokens, output_tokens, model):
    pricing_2024 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 5/1e6, "output": 15/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
        "claude-3-opus": {"input": 15/1e6, "output": 75/1e6},
        "claude-3.5-sonnet": {"input": 3/1e6, "output": 15/1e6},
    }
    pricing_2025 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 2.5/1e6, "output": 10/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
    }
    
    prices = pricing_2025[model]
    cost = (input_tokens * prices["input"]) + (output_tokens * prices["output"])
    return round(cost, 6)

In [3]:
llm_model = "gpt-4o-2024-08-06"
tools = "all"

agent = MDCrow(
    agent_type="Structured", 
    model=llm_model, 
    top_k_tools=tools,
    streaming=False,
    verbose=True,
    modifysim_no_run=True,
    use_memory=True,
)

In [4]:
prompt = """I want to simulate the protein 2F4K at 350 K. Set up and 
run the simulation for a few timesteps to ensure everything is working 
correctly and set up properly."""

with get_openai_callback() as cb:
    print(agent.run(prompt, callbacks=[cb]))
    print(cb)
    print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))



> Entering new AgentExecutor chain...
Thought: To simulate the protein 2F4K at 350 K, I need to download the PDB file for 2F4K, set up the simulation parameters with the specified temperature, and run a short simulation to ensure everything is working correctly. First, I'll download the PDB file for 2F4K.

Action:
```json
{
  "action": "PDBFileDownloader",
  "action_input": "2F4K"
}
```PDB file found with this ID: 2F4K

Observation: Succeeded. Downloaded the PDB file:2F4K_140234
Thought:Action:
```json
{
  "action": "SetUpandRunFunction",
  "action_input": {
    "pdb_id": "2F4K_140234",
    "forcefield_files": ["amber99sb.xml", "tip3p.xml"],
    "save": true,
    "system_params": {
      "nonbondedMethod": "NoCutoff",
      "constraints": "None",
      "rigidWater": false
    },
    "integrator_params": {
      "integrator_type": "LangevinMiddle",
      "Temperature": "350 * kelvin",
      "Friction": "1.0 / picoseconds",
      "Timestep": "0.002 * picoseconds"
    },
    "simulation

In [ ]:
prompt = """I want to simulate the protein 2F4K at 350 K. Set up properly and 
run the simulation for some timesteps to ensure everything is working 
correctly."""

with get_openai_callback() as cb:
    print(agent.run(prompt, callbacks=[cb]))
    print(cb)
    print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))

In [11]:
prompt = """We just did the brief simulation setup. Now, check the box 
size and ensure it's sufficient for unfolding of this protein."""

with get_openai_callback() as cb:
    print(agent.run(prompt, callbacks=[cb]))
    print(cb)
    print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))



> Entering new AgentExecutor chain...
Thought: To check the box size and ensure it's sufficient for the unfolding of the protein, I need to calculate the radius of gyration of the protein. This will give an idea of the protein's size and help determine if the box is large enough for unfolding. I will use the RadiusofGyrationTool for this purpose. 

Action:
```
{
  "action": "RadiusofGyrationTool",
  "action_input": {
    "traj_file": "trajectory_file_id", 
    "top_file": "topology_file_id"
  }
}
```

Observation: Error loading traj: Topology File ID 'topology_file_id' not found in PathRegistry
Thought:To proceed, I need to list the available paths in the registry to identify the correct topology and trajectory file IDs. This will help me access the necessary files for calculating the radius of gyration.

Action:
```json
{
  "action": "ListRegistryPaths",
  "action_input": {
    "paths": "paths"
  }
}
```
Observation: Succeeded. Files found in registry: 2F4K_140234: PDB file download

<Figure size 640x480 with 0 Axes>

In [ ]:
# enable run within modifysim tool
agent = MDCrow(
    agent_type="Structured", 
    model=llm_model, 
    top_k_tools=tools,
    streaming=False,
    verbose=True,
    modifysim_no_run=True, # whoops, should be False
    use_memory=True,
)

In [ ]:
# 2nd attempt, enabling run within modifysim tool 
# NOTE: this failed because i forgot to replace prompt variable
prompt_partA = """I want to simulate the protein 2F4K at 350 K. Set up using
appropriate parameters for this protein simulation 
(you can collect information as needed). Run the simulation for some timesteps 
to ensure everything is working correctly. """

with get_openai_callback() as cb:
    print(agent.run(prompt, callbacks=[cb]))
    print(cb)
    print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))



> Entering new AgentExecutor chain...
Thought: To simulate the protein 2F4K at 350 K, I need to download the PDB file for 2F4K, set up the simulation parameters including the temperature, and run a short simulation to ensure everything is working correctly. I will start by downloading the PDB file for 2F4K.

Action:
```
{
  "action": "PDBFileDownloader",
  "action_input": "2F4K"
}
```PDB file found with this ID: 2F4K

Observation: Succeeded. Downloaded the PDB file:2F4K_145728
Thought:To proceed with setting up and running the simulation for the protein 2F4K at 350 K, I need to configure the simulation parameters, including the temperature, and execute a short simulation to verify that everything is functioning correctly. I will use the downloaded PDB file to set up the simulation.

Action:
```
{
  "action": "SetUpandRunFunction",
  "action_input": {
    "pdb_id": "2F4K_145728",
    "forcefield_files": ["amber99sb.xml", "tip3p.xml"],
    "save": true,
    "system_params": {
      "no

In [ ]:
# failed prompt B
prompt_partB="""We did a basic simulation on 2F4K at 350 K earlier. First, estimate the box 
size we may need to observe unfolding of 2F4K. Then use that information to modify the 
simulation script as needed.""" # it should add approrpriate amount of padding to ensure the box is sufficient.

with get_openai_callback() as cb:
    print(agent.run(prompt, callbacks=[cb]))
    print(cb)
    print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))



> Entering new AgentExecutor chain...
Thought: To simulate the protein 2F4K at 350 K, I need to download the PDB file for 2F4K, set up the simulation parameters including the temperature, and run a short simulation to ensure everything is working correctly. I will start by downloading the PDB file for 2F4K.

Action:
```json
{
  "action": "PDBFileDownloader",
  "action_input": "2F4K"
}
```PDB file found with this ID: 2F4K

Observation: Succeeded. Downloaded the PDB file:2F4K_151043
Thought:To proceed with the simulation of the protein 2F4K at 350 K, I need to set up the simulation parameters, including the temperature, and run a short simulation to ensure everything is working correctly. I will use the downloaded PDB file to set up and run the simulation.

Action:
```json
{
  "action": "SetUpandRunFunction",
  "action_input": {
    "pdb_id": "2F4K_151043",
    "forcefield_files": ["amber99sb.xml", "tip3p.xml"],
    "save": true,
    "system_params": {
      "nonbondedMethod": "NoCutof

In [26]:
prompt_partB="""Estimate the box size we may need to observe unfolding of 2F4K."""

with get_openai_callback() as cb:
    print(agent.run(prompt_partB, callbacks=[cb]))
    print(cb)
    print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))



> Entering new AgentExecutor chain...
Thought: To estimate the box size needed to observe the unfolding of the protein 2F4K, I need to first obtain the PDB file for 2F4K. Then, I can analyze the structure to determine its dimensions and add some buffer space to allow for unfolding. This will help in estimating the appropriate box size for molecular dynamics simulations.

Action:
```json
{
  "action": "PDBFileDownloader",
  "action_input": {
    "query": "2F4K"
  }
}
```PDB file found with this ID: 2F4K

Observation: Succeeded. Downloaded the PDB file:2F4K_151729
Thought:To estimate the box size needed for observing the unfolding of the protein 2F4K, I should analyze the structure to determine its dimensions. This will involve calculating the radius of gyration or using the moment of inertia to understand the spatial extent of the protein. Then, I can add a buffer to these dimensions to allow for unfolding.

Action:
```json
{
  "action": "MomentOfInertia",
  "action_input": {
    "top

<Figure size 640x480 with 0 Axes>

In [32]:
# add memory context about box size estimate
agent = MDCrow(
    agent_type="Structured", 
    model=llm_model, 
    top_k_tools=tools,
    streaming=False,
    verbose=True,
    modifysim_no_run=False,
    use_memory=True,
    run_id="2W3LCCCA",
    ckpt_dir="ckpt_52"
)

In [36]:
prompt_partC="""Earlier, we did a basic simulation on 2F4K at 350 K. First, estimate
the box size we may need to accomodate the unfolding of 2F4K. Use that information to 
modify the simulation script to solvate and add a specific amount of padding.""" # "make an educated estimate"

with get_openai_callback() as cb:
    print(agent.run(prompt_partC, callbacks=[cb]))
    print(cb)
    print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))



> Entering new AgentExecutor chain...
Thought: To estimate the box size needed to accommodate the unfolding of protein 2F4K, we should first calculate the radius of gyration, which gives an idea of the protein's size in its folded state. Then, we can add a specific amount of padding to ensure enough space for potential unfolding. After estimating the box size, we can modify the simulation script to include solvation and the specified padding.

Action:
```
{
  "action": "RadiusofGyrationTool",
  "action_input": {
    "traj_file": "2F4K_trajectory_file_id",
    "top_file": "2F4K_topology_file_id"
  }
}
```
Observation: Error loading traj: Topology File ID '2F4K_topology_file_id' not found in PathRegistry
Thought:To proceed with estimating the box size for the unfolding of protein 2F4K, I need to ensure that the correct topology and trajectory files are available. Since there was an error indicating that the topology file ID was not found, I should first list the available paths to iden

<Figure size 640x480 with 0 Axes>